In [1]:
import pandas as pd, numpy as np
from scipy import stats

df = pd.read_csv("../data/clean/gaming_clean.csv")

In [6]:
#Compare spend across groups
def compare_groups(df, group, value):
    groups = [g[value].values for _, g in df.groupby(group)]
    F, p_anova = stats.f_oneway(*groups)
    H, p_kw = stats.kruskal(*groups)
    gm = df[value].mean()
    eta2 = sum(len(g)*(g.mean()-gm)**2 for g in groups) / ((df[value]-gm)**2).sum()
    return {"group": group, "value": value, "p_anova": round(p_anova,3),
            "p_kruskal": round(p_kw,3), "eta_squared": round(eta2,6)}

rows = [compare_groups(df, g, v)
        for g in ["platform","genre","age_group","gender","state"]
        for v in ["spend_inr","daily_hours"]]
pd.DataFrame(rows)

,group,value,p_anova,p_kruskal,eta_squared
0,platform,spend_inr,0.235,0.235,0.000004
1,platform,daily_hours,0.305,0.141,0.000004
2,genre,spend_inr,0.615,0.615,0.000003
3,genre,daily_hours,0.691,0.648,0.000002
4,age_group,spend_inr,0.008,0.008,0.000012
5,age_group,daily_hours,0.075,0.178,0.000007
6,gender,spend_inr,0.514,0.514,0.000001
7,gender,daily_hours,0.303,0.136,0.000002
8,state,spend_inr,0.472,0.471,0.000010
9,state,daily_hours,0.144,0.261,0.000015


Spend and daily hours do not differ meaningfully by platform, genre, gender or state. All eta-squared values are below 0.00002, far under the 0.01 threshold for a small effect. Age group shows a nominally significant spend difference (p = 0.008), but the gap is only about ₹20 (eta² = 0.000012), which is not practically meaningful.

In [7]:
#Compare esports and churn flags
def compare_flag(df, flag, value):
    a, b = df[df[flag]][value], df[~df[flag]][value]
    t, p_t = stats.ttest_ind(a, b, equal_var=False)
    u, p_u = stats.mannwhitneyu(a, b)
    d = (a.mean()-b.mean()) / np.sqrt((a.var()+b.var())/2)   # Cohen's d
    return {"flag": flag, "value": value, "mean_true": round(a.mean(),2),
            "mean_false": round(b.mean(),2), "p_ttest": round(p_t,3),
            "p_mannwhitney": round(p_u,3), "cohens_d": round(d,4)}

pd.DataFrame([compare_flag(df, f, v)
              for f in ["esports_participant","churn_risk"]
              for v in ["spend_inr","daily_hours","age"]])

,flag,value,mean_true,mean_false,p_ttest,p_mannwhitney,cohens_d
0,esports_participant,spend_inr,3756.09,3749.52,0.326,0.326,0.0030
1,esports_participant,daily_hours,2.20,2.20,0.776,0.651,-0.0009
2,esports_participant,age,29.52,29.50,0.503,0.503,0.0021
3,churn_risk,spend_inr,3748.29,3750.87,0.622,0.622,-0.0012
4,churn_risk,daily_hours,2.20,2.20,0.645,0.395,-0.0011
5,churn_risk,age,29.49,29.51,0.642,0.643,-0.0011


Esports participants and non-participants are virtually identical in spend (₹3,756 vs ₹3,750), daily hours (2.20 vs 2.20) and age (29.5 vs 29.5); Cohen's d is at most 0.003. Churn-risk players also match the others on all three measures (d ≤ 0.0012). No difference is statistically or practically meaningful.

In [8]:
#Test relationships between categories
def cramers_v(df, a, b):
    ct = pd.crosstab(df[a], df[b])
    chi, p, _, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi / (ct.values.sum() * (min(ct.shape)-1)))
    return {"var1": a, "var2": b, "p_value": round(p,3), "cramers_v": round(v,4)}

pairs = [("gender","genre"),("platform","genre"),("state","platform"),
         ("esports_participant","genre"),("esports_participant","platform")]
pd.DataFrame([cramers_v(df, a, b) for a, b in pairs])

,var1,var2,p_value,cramers_v
0,gender,genre,0.037,0.0029
1,platform,genre,0.679,0.0018
2,state,platform,0.383,0.0032
3,esports_participant,genre,0.859,0.0011
4,esports_participant,platform,0.665,0.0013


Esports participants and non-participants are virtually identical in spend (₹3,756 vs ₹3,750), daily hours (2.20 vs 2.20) and age (29.5 vs 29.5); Cohen's d is at most 0.003. Churn-risk players also match the others on all three measures (d ≤ 0.0012). No difference is statistically or practically meaningful.

In [9]:
#Churn rate by segment
for seg in ["platform","genre","age_group","gender","state","signup_year","esports_participant"]:
    print((df.groupby(seg).churn_risk.mean()*100).round(1), "\n")

platform
Cloud Gaming             21.9
Console (PS5/Xbox)       22.0
Mobile (Android/iOS)     22.0
PC (Steam/Epic Games)    22.0
Name: churn_risk, dtype: float64 

genre
Action-RPG & Souls-like                      21.9
Battle Royale (BGMI/Free Fire)               22.0
Casual, Puzzle & Strategy                    22.2
Cricket & Sports Simulation                  21.9
Esports & Tactical Shooters (Valorant/CS)    22.0
Name: churn_risk, dtype: float64 

age_group
18-24       22.0
25-34       22.1
35-47       21.9
Under 18    21.9
Name: churn_risk, dtype: float64 

gender
Female    22.0
Male      21.9
Other     22.3
Name: churn_risk, dtype: float64 

state
Bihar             22.0
Delhi NCR         22.0
Gujarat           22.2
Karnataka         22.0
Madhya Pradesh    21.8
Maharashtra       21.9
Rajasthan         22.1
Tamil Nadu        22.0
Telangana         21.8
Uttar Pradesh     21.9
West Bengal       22.0
Name: churn_risk, dtype: float64 

signup_year
2021    22.1
2022    22.0
2023    22.0


Churn risk is about 22% in every segment, with segment rates ranging only from 21.8% to 22.3%. No platform, genre, age group, gender, state, signup cohort or esports status identifies higher-risk players, so churn cannot be targeted using these variables.

NOTE: About 35 hypothesis tests were run, so one or two p-values below 0.05 are expected by chance. With a Bonferroni-adjusted threshold of about 0.0014, none of the results is significant. Because the data is synthetic, a lack of structure is plausible and consistent with randomly generated attributes